# Boosting

Boosting trains weak learners in sequence, each one focusing on the errors of the last. We compare AdaBoost, gradient boosting, and XGBoost with the earlier models.

## Imports and loading helper

In [ ]:
# Standard imports for the whole course
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Find the Datasets folder whether the notebook is run from the
# project root or from inside the notebooks folder.
from pathlib import Path
for candidate in [Path("Datasets"), Path("../Datasets")]:
    if candidate.exists():
        DATA = candidate
        break

def load(name):
    """Load a CSV from the Datasets folder by file name."""
    return pd.read_csv(DATA / name)

print("Datasets are in:", DATA.resolve())
print(sorted(p.name for p in DATA.glob("*.csv")))

## Preparing the customer data

In [ ]:
df = load("Naive_Bayes_Decision_RF_Tree_Boosting_User_Data.csv")
X = df.iloc[:, [2, 3]].values
y = df.iloc[:, 4].values

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## AdaBoost

A weighted vote of many small stumps, with more weight on the hard examples.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score

ada = AdaBoostClassifier(n_estimators=50, learning_rate=0.1, random_state=0)
ada.fit(X_train, y_train)
print("AdaBoost:", accuracy_score(y_test, ada.predict(X_test)))

## Gradient Boosting

Each new tree is fitted to the errors left by the current ensemble.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=0).fit(X_train, y_train)
print("Gradient boosting:", accuracy_score(y_test, gb.predict(X_test)))

## XGBoost

A fast, well engineered form of gradient boosting. If the package is not installed, the cell simply reports that.

In [ ]:
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(eval_metric="logloss").fit(X_train, y_train)
    print("XGBoost:", accuracy_score(y_test, xgb.predict(X_test)))
except ImportError:
    print("xgboost is not installed; run pip install xgboost")

## Comparing the models

On an easy problem the strong models cluster together in accuracy, so speed and convenience decide between them.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

dt = DecisionTreeClassifier(criterion="entropy", random_state=0).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=10, criterion="entropy",
                            random_state=0).fit(X_train, y_train)

names = ["Decision tree", "Random forest", "AdaBoost", "Gradient boosting"]
scores = [accuracy_score(y_test, m.predict(X_test))
          for m in (dt, rf, ada, gb)]

plt.bar(names, scores, color="#55A868")
plt.ylim(0.6, 1.02); plt.ylabel("Accuracy")
plt.xticks(rotation=15); plt.show()